### Image Clustering - Improving Validation Set

A naive training/validation split will contain many near-duplicate images, as well as images that are not near duplicates but are very similar and are taken from the same drone on the same day in the same location. This leads to severely inflated validation set performance and a large degradation on the test set. To improve the validation set, I decided to use a deep learning based approach to group the images in the training set into clusters based on visual similarity. My first approach used an ImageNet-trained ResNet50 model backbone to extract features. I then used HBDSCAN to group the images into clusters. This approach did separate images into visually distinct, but it didn't do a good enough job given the size and complexity of the dataset. I tried manually fixing many of the outliers and misclassified images, but it proved infeasible. So, I did some research to try to find what a better approach would be. What the internet seems to recommend is to use the self-supervised transformer-based DINOv2 model to extract features, as this should give far improved results especially for large aerial drone images. Below, I implement an image clustering algorithm using DINOv2 and visualize the results.

In [ ]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from hdbscan import HDBSCAN
from torch.utils.data import Dataset as TorchDataset, DataLoader
import albumentations as A
import cv2
from pathlib import Path
import shutil
import matplotlib.pyplot as plt

In [ ]:
@dataclass
class Config:
    train_csv_filepath: str
    images_root_folder: str
    device: str

    # noinspection PyAttributeOutsideInit
    def init(self):
        self.image_height = 518
        self.image_width = 924
        self.image_dims = (self.image_height, self.image_width)
        self.image_transforms = A.Compose([
            A.Resize(height=self.image_height, width=self.image_width, interpolation=cv2.INTER_AREA),
            A.Normalize(imagenet_mean_tuple, imagenet_std_tuple),
            A.ToTensorV2(),
        ])
        self.model_repo = 'facebookresearch/dinov2'
        self.model_name = 'dinov2_vitb14'
        self.min_cluster_size = 3
        self.min_samples = 3
        self.cluster_selection_epsilon = 45.0
        self.cluster_selection_method = 'eom'
        self.pca_components = None
        self.separate_outliers = True
        self.num_workers = 4 if self.device == 'cuda' else 0
        self.pin_memory = self.num_workers > 0
        self.batch_size = 16
        self.image_ids = pd.read_csv(self.train_csv_filepath)['ImageID'].to_numpy()
        self.embeddings_file = f'{self.images_root_folder}embeddings.npy'
        self.cluster_labels_file = f'{self.images_root_folder}cluster_labels.csv'

config: Config = None

In [ ]:
local_config = Config(
    train_csv_filepath='data/train.csv',
    images_root_folder='data/train_images/',
    device='cpu',
)

In [ ]:
imagenet_mean_tuple = (0.485, 0.456, 0.406)
imagenet_std_tuple = (0.229, 0.224, 0.225)

In [ ]:
class ClusterDataset(TorchDataset):
    def __len__(self):
        return len(config.image_ids)

    def __getitem__(self, idx):
        image_id = config.image_ids[idx]
        image = cv2.cvtColor(cv2.imread(f'{config.images_root_folder}{image_id}.jpg'), cv2.COLOR_BGR2RGB)
        return config.image_transforms(image)['image']

In [ ]:
def generate_image_clusters():
    dataset = ClusterDataset()
    loader = DataLoader(dataset, shuffle=False, batch_size=config.batch_size, num_workers=config.num_workers, pin_memory=config.pin_memory)

    model = torch.hub.load(config.model_repo, config.model_name).to(config.device)
    model.eval()

    embeddings = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(config.device)
            batch_embeddings = model(batch)
            embeddings.append(batch_embeddings.cpu().numpy())

    embeddings = np.vstack(embeddings)
    np.save(config.embeddings_file, embeddings)
    print(f'Embeddings saved to {config.embeddings_file}')
    embeddings_dim = embeddings.shape[1]

    scaler = StandardScaler()
    embeddings = scaler.fit_transform(embeddings)

    if config.pca_components is not None:
        effective_pca = min(config.pca_components, embeddings_dim)
        pca = PCA(n_components=effective_pca)
        embeddings = pca.fit_transform(embeddings)
        explained_var = pca.explained_variance_ratio_.sum()
        print(f'PCA: {effective_pca} components explain {explained_var:.1%} of variance')

    hdbscan = HDBSCAN(
        min_cluster_size=config.min_cluster_size,
        min_samples=config.min_samples,
        cluster_selection_epsilon=config.cluster_selection_epsilon,
        cluster_selection_method=config.cluster_selection_method,
        metric='euclidean',
    )
    cluster_labels = hdbscan.fit_predict(embeddings)
    unique, counts = np.unique(cluster_labels, return_counts=True)
    sizes = sorted(counts, reverse=True)
    print(f'Cluster sizes (sorted): {sizes[:20]}{'...' if len(sizes) > 20 else ''}')
    print(f'Min: {min(sizes)}, Max: {max(sizes)}, Mean: {np.mean(sizes):.1f}')

    df = pd.DataFrame({'id': config.image_ids, 'cluster': cluster_labels})
    df.to_csv(config.cluster_labels_file, index=False)
    print(f'Cluster labels saved to {config.cluster_labels_file}')

In [ ]:
def group_images_into_cluster_folders(images_folder: str, cluster_labels_file: str):
    cluster_labels_df = pd.read_csv(f'{images_folder}{cluster_labels_file}')
    unique_clusters = cluster_labels_df['cluster'].unique()
    for cluster_id in unique_clusters:
        Path(f'{images_folder}cluster_{cluster_id}').mkdir(exist_ok=True)
    moved = 0
    for image_id, cluster_id in cluster_labels_df.itertuples(index=False):
        shutil.move(f'{images_folder}{image_id}.jpg', f'{images_folder}cluster_{cluster_id}/{image_id}.jpg')
        moved += 1
    print(f'Moved {moved} images into {len(unique_clusters)} folders.')

In [ ]:
def ungroup_images_in_folder(images_folder: str):
    images_folder = Path(images_folder)
    subfolders = [item for item in images_folder.iterdir() if item.is_dir()]
    moved = 0
    deleted = 0
    for subfolder in subfolders:
        for file in subfolder.iterdir():
            shutil.move(str(file), str(images_folder / file.name))
            moved += 1
        subfolder.rmdir()
        deleted += 1
    print(f"Moved {moved} files back to the main folder. Deleted {deleted} subfolders.")

In [ ]:
config = local_config
config.init()
generate_image_clusters()